In [ ]:
from pathlib import Path
from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache

data_dir = Path("../data/ecephys_cache")
data_dir.mkdir(parents=True, exist_ok=True)

cache = EcephysProjectCache.from_warehouse(manifest=data_dir / "manifest.json")

sessions = cache.get_session_table()
print(sessions.shape)
sessions.head()

In [ ]:
bo = sessions[sessions.session_type == "brain_observatory_1.1"]
wt = bo[bo.full_genotype == "wt/wt"].sort_values("unit_count", ascending=False)
wt[["unit_count", "channel_count", "probe_count", "age_in_days", "sex"]].head(10)

In [ ]:
session_id = 732592105
session = cache.get_session_data(session_id)

stim = session.stimulus_presentations
print(stim.stimulus_name.value_counts())

sg = stim[stim.stimulus_name == "static_gratings"]
print(sg.shape)
sg[["start_time", "stop_time", "orientation", "spatial_frequency", "phase"]].head()
print(sg.orientation.value_counts())

In [ ]:
import numpy as np
import pandas as pd

# Trials: drop blanks, keep what we need
sg = stim[stim.stimulus_name == "static_gratings"].copy()
sg = sg[sg.orientation != "null"]
sg["orientation"] = sg["orientation"].astype(float)
print(sg.shape)

# Units: quality filter + visual cortex
units = session.units
print(units.ecephys_structure_acronym.value_counts())

In [ ]:
print(units.columns.tolist())
print(len(units))

In [ ]:
vis = units[
    units.ecephys_structure_acronym.str.startswith("VIS", na=False)
    & (units.isi_violations < 0.5)
    & (units.amplitude_cutoff < 0.1)
    & (units.presence_ratio > 0.9)
    & (units.firing_rate > 0.5)
]
print(len(vis), "units kept")
print(vis.ecephys_structure_acronym.value_counts())

In [ ]:
window = 0.25 # seconds after stimulus onset
bin_edges = np.array([0.0, window])

counts = session.presentationwise_spike_counts(
    bin_edges=bin_edges,
    stimulus_presentation_ids=sg.index.values,
    unit_ids=vis.index.values,
)
print(counts.dims, counts.shape)

In [ ]:
x = counts.squeeze("time_relative_to_stimulus_onset").values # (n_trials, n_units)
y = sg.loc[counts.stimulus_presentation_id.values, "orientation"].values

print(x.shape, y.shape)
print(np.unique(y, return_counts=True))

In [ ]:
print(np.isnan(x).any(), x.mean(), x.min(), x.max())
print("silent units:", (x.sum(axis=0) == 0).sum())

In [ ]:
import matplotlib.pyplot as plt

orients = np.unique(y)
# mean spike count per orientation: shape (n_orientations, n_units)
tuning = np.array([x[y == o].mean(axis=0) for o in orients])

# pick the 6 units with the most orientation-selective means (largest range)
sel = np.argsort(tuning.max(axis=0) - tuning.min(axis=0))[-6:]

fig, axes = plt.subplots(2, 3, figsize=(10, 6), sharex=True)
for ax, u in zip(axes.ravel(), sel):
    sem = np.array([x[y == o, u].std(ddof=1) / np.sqrt((y == o).sum()) for o in orients])
    ax.errorbar(orients, tuning[:, u], yerr=sem, marker="o")
    ax.set_title(f"unit {vis.index[u]} ({vis.ecephys_structure_acronym.iloc[u]})")
    ax.set_xlabel("orientation (deg)")
    ax.set_ylabel("spike / 250 ms")
plt.tight_layout()

plt.savefig("../results/tuning_curves.png", dpi=200, bbox_inches="tight") 
plt.show()

In [ ]:
pref = tuning.argmax(axis=0)
order = np.argsort(pref)
norm = (tuning - tuning.min(axis=0)) / (tuning.max(axis=0) - tuning.min(axis=0) + 1e-9)

plt.figure(figsize=(5, 7))
plt.imshow(norm[:, order].T, aspect="auto", cmap="viridis")
plt.xticks(range(len(orients)), orients.astype(int))
plt.xlabel("orientation (deg)")
plt.ylabel("units (sorted by preferred orientation)")
plt.colorbar(label="normalised mean response")

plt.savefig("../results/tuning_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
import time
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Scaling lives inside the pipeline, so it is fit on training folds only
clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.01, max_iter=2000),
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# Real decoding accuracy
t0 = time.time()
scores = cross_val_score(clf, x, y, cv=cv, scoring="accuracy", n_jobs=-1)
print("fold accuracies:", np.round(scores, 3))
print(f"mean accuracy: {scores.mean():.3f} ± {scores.std():.3f}    ({time.time()-t0:.1f} s)")

In [ ]:
# Shuffled-label control
rng = np.random.default_rng(0)
null = []
for i in range(20):
    y_shuf = rng.permutation(y)
    null.append(cross_val_score(clf, x, y_shuf, cv=cv).mean())
null = np.array(null)
print(f"shuffled: {null.mean():.3f} ± {null.std():.3f}    (1/6 = {1/6:.3f})")

In [ ]:
from sklearn.model_selection import KFold

cv_block = KFold(n_splits=5, shuffle=False)
scores_block = cross_val_score(clf, x, y, cv=cv_block, n_jobs=-1)
print("blocked folds:", np.round(scores_block, 3), f"mean {scores_block.mean():.3f}")

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = cross_val_predict(clf, x, y, cv=cv)
cm = confusion_matrix(y, y_pred, labels=orients, normalize="true")
ConfusionMatrixDisplay(cm, display_labels=orients.astype(int)).plot(cmap="Blues")

plt.savefig("../results/confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {"logisticregression__C": [1e-4, 1e-3, 1e-2, 1e-1, 1]}
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

search = GridSearchCV(clf, param_grid, cv=inner, scoring="accuracy", n_jobs=-1)

t0 = time.time()
nested = cross_val_score(search, x, y, cv=outer)
print("nested fold accuracies:", np.round(nested, 3))
print(f"nested accuracy: {nested.mean():.3f} ± {nested.std():.3f}     ({time.time()-t0:.1f} s)")

# which C wins on the full data
search.fit(x, y)
print("best C on all data:", search.best_params_)
print(np.round(search.cv_results_["mean_test_score"], 3))

In [ ]:
sizes = [1, 2, 5, 10, 20, 50, 100, 200, x.shape[1]]
n_repeats = 20
tune = True # set True to re-tune C for every subset (slower)

rng = np.random.default_rng(0)
rows = []
t0 = time.time()
for n in sizes:
    reps = 1 if n == x.shape[1] else n_repeats
    for r in range(reps):
        idx = rng.choice(x.shape[1], size=n, replace=False)
        model = GridSearchCV(clf, param_grid, cv=inner, n_jobs=-1) if tune else clf
        acc = cross_val_score(model, x[:, idx], y, cv=cv, n_jobs=-1).mean()
        rows.append({"n_units": n, "repeat": r, "accuracy": acc})
    print(f"n={n:4d}   done   ({time.time()-t0:.0f} s)")

res = pd.DataFrame(rows)
res.to_csv("../results/decoding_by_n.csv", index=False)

summ = res.groupby("n_units")["accuracy"].agg(["mean", "std"])
print(summ.round(3))

plt.errorbar(summ.index, summ["mean"], yerr=summ["std"], marker="o")
plt.axhline(1/6, ls="--", color="gray", label="chance (1/6)")
plt.xscale("log")
plt.xlabel("number of units")
plt.ylabel("decoding accuracy")
plt.legend()
plt.show()